# Klasifikasi Email Spam/Ham
## Notebook Gabungan: Preprocessing + Modeling (Google Colab)
### Dataset: `spam_ham_dataset.csv`

Notebook ini dibuat berdasarkan hasil investigasi sebelumnya terhadap `enron_spam_data.csv` yang ternyata
punya bug data leakage parah (kelas spam cuma berisi 6 email unik yang diduplikasi ribuan kali). Dataset
`spam_ham_dataset.csv` ini sudah divalidasi lebih dulu:

| Cek | Hasil |
|---|---|
| Total baris | 5.171 |
| Distribusi | ham: 3.672, spam: 1.499 |
| Duplikat baris penuh | 0 |
| Teks unik spam | 1.462 dari 1.499 (97.5%) |
| Teks unik ham | 3.531 dari 3.672 (96.2%) |

Data ini sehat dan siap dipakai.

### 0. Setup & Upload Dataset

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import pandas as pd
import numpy as np
import re
import time

In [ ]:
# Sesuaikan path ini dengan lokasi file di Google Drive kamu
file_path = '/content/drive/MyDrive/Semester 4/Machine Learning/Final Project/data/spam_ham_dataset.csv'
df = pd.read_csv(file_path)
df.head()

### 1. Inspeksi Awal Data

In [ ]:
print("Dimensi dataset:", df.shape)
print("-" * 30)
df.info()
print()
print("Distribusi label:")
print(df['label'].value_counts())

### 2. Data Cleaning: Missing Value & Kolom Tidak Perlu

Kolom `Unnamed: 0` cuma index bawaan CSV, dan `label_num` redundan dengan `label` (0/1 encoding).
Kita pertahankan `label` (string) sebagai target utama.

In [ ]:
df_cleaned = df.drop(['Unnamed: 0', 'label_num'], axis=1)
df_cleaned = df_cleaned.dropna(subset=['text'])
print("Dimensi setelah pembersihan awal:", df_cleaned.shape)

### 3. Menangani Data Duplikat (**WAJIB — pelajaran dari investigasi sebelumnya**)

> Ini langkah paling penting untuk mencegah *data leakage*. JANGAN pernah menonaktifkan baris ini,
walau kelihatan mengurangi jumlah data — duplikat yang dibiarkan akan membuat model "menghafal"
alih-alih belajar pola, dan menghasilkan angka akurasi yang menyesatkan (kelihatan tinggi padahal
model gagal total di dunia nyata).

In [ ]:
jumlah_duplikat = df_cleaned.duplicated(subset=['text']).sum()
print(f"Ditemukan {jumlah_duplikat} baris teks duplikat.")

df_cleaned = df_cleaned.drop_duplicates(subset=['text']).reset_index(drop=True)

print("Dimensi setelah dedup:", df_cleaned.shape)
print(df_cleaned['label'].value_counts())

### 3b. Sanity Check: Pastikan Tidak Ada Konflik Label (**BARU**)

Cek apakah ada teks yang sama persis tapi diberi label berbeda (tanda dataset bermasalah, seperti
yang ditemukan di dataset sebelumnya).

In [ ]:
konflik_mask = df_cleaned.duplicated(subset=['text'], keep=False)
print(f"Jumlah baris dengan teks sama tapi label konflik: {konflik_mask.sum()}")
# Harusnya 0 karena kita sudah dedup berdasarkan text di atas

### 4. Text Preprocessing (Case Folding, Hapus Simbol, Stopwords)

In [ ]:
def bersihkan_teks(teks):
    teks = teks.lower()
    teks = re.sub(r'[^a-z\s]', ' ', teks)
    teks = re.sub(r'\s+', ' ', teks).strip()
    return teks

df_cleaned['clean_text'] = df_cleaned['text'].apply(bersihkan_teks)
df_cleaned[['text', 'clean_text']].head()

In [ ]:
import nltk
from nltk.corpus import stopwords

nltk.download('stopwords')
stop_words = set(stopwords.words('english'))

def hapus_stopword(teks):
    kata_kata = teks.split()
    kata_bersih = [kata for kata in kata_kata if kata not in stop_words]
    return ' '.join(kata_bersih)

df_cleaned['final_text'] = df_cleaned['clean_text'].apply(hapus_stopword)
df_cleaned[['clean_text', 'final_text']].head()

### 5. Menyimpan Data Bersih (Opsional, untuk arsip)

In [ ]:
df_final = df_cleaned[['final_text', 'label']].copy()
df_final.columns = ['text', 'label']
df_final = df_final[df_final['text'].str.strip() != '']

save_path = '/content/drive/MyDrive/Semester 4/Machine Learning/Final Project/data/cleaned_spam_ham_dataset.csv'
df_final.to_csv(save_path, index=False)

print("Data bersih disimpan di:", save_path)
print("Dimensi akhir:", df_final.shape)
print(df_final['label'].value_counts())

---
## MODELING

### 6. Memisahkan Fitur (X) dan Target (y), lalu Split Train/Test

In [ ]:
from sklearn.model_selection import train_test_split

X = df_final['text']
y = df_final['label']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print("Jumlah Data Latih:", len(X_train))
print("Jumlah Data Uji  :", len(X_test))
print()
print("Distribusi label di Data Latih:")
print(y_train.value_counts())
print()
print("Distribusi label di Data Uji:")
print(y_test.value_counts())

### 7. Feature Extraction (TF-IDF)

`fit` hanya pada Data Latih untuk mencegah *data leakage*, lalu `transform` ke Data Latih dan Data Uji.

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf = TfidfVectorizer(max_features=5000)
X_train_tfidf = tfidf.fit_transform(X_train)
X_test_tfidf = tfidf.transform(X_test)

print("Bentuk matriks Data Latih:", X_train_tfidf.shape)
print("Bentuk matriks Data Uji:", X_test_tfidf.shape)

### 8. Pembangunan dan Pelatihan Model

Melatih dan membandingkan 2 algoritma:
1. Naive Bayes (MultinomialNB)
2. Support Vector Machine (LinearSVC) dengan `class_weight='balanced'` (data sedikit imbalance: 71% ham, 29% spam)

In [ ]:
from sklearn.naive_bayes import MultinomialNB
from sklearn.svm import LinearSVC

nb_model = MultinomialNB()
svm_model = LinearSVC(random_state=42, class_weight='balanced')

print("Melatih Naive Bayes...")
start = time.time()
nb_model.fit(X_train_tfidf, y_train)
print(f"Selesai ({time.time()-start:.2f} detik)\n")

print("Melatih SVM...")
start = time.time()
svm_model.fit(X_train_tfidf, y_train)
print(f"Selesai ({time.time()-start:.2f} detik)")

### 9. Evaluasi Model

In [ ]:
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

nb_pred = nb_model.predict(X_test_tfidf)
svm_pred = svm_model.predict(X_test_tfidf)

print(f"Akurasi Naive Bayes: {accuracy_score(y_test, nb_pred)*100:.2f}%")
print(f"Akurasi SVM        : {accuracy_score(y_test, svm_pred)*100:.2f}%\n")

print("-"*50)
print("Detail Performa NB:")
print(classification_report(y_test, nb_pred))

print("-"*50)
print("Detail Performa SVM:")
print(classification_report(y_test, svm_pred))

In [ ]:
import matplotlib.pyplot as plt
from sklearn.metrics import ConfusionMatrixDisplay

fig, axes = plt.subplots(1, 2, figsize=(12,5))
ConfusionMatrixDisplay.from_predictions(y_test, nb_pred, ax=axes[0], colorbar=False)
axes[0].set_title('Naive Bayes')
ConfusionMatrixDisplay.from_predictions(y_test, svm_pred, ax=axes[1], colorbar=False)
axes[1].set_title('SVM')
plt.tight_layout()
plt.show()

### 10. Sanity Check dengan Contoh Baru (di luar dataset)

Wajib dilakukan sebelum menyimpan model — pastikan model bisa mengenali pola spam yang jelas,
bukan cuma menghafal dataset.

In [ ]:
contoh_teks = [
    "congratulations you have won a free lottery prize click here now claim reward money",
    "buy cheap viagra online discount pills no prescription needed order now",
    "hi john please find attached report last month let me know if you have questions thanks",
    "meeting scheduled tomorrow morning please bring your laptop and notes",
    "urgent your account has been suspended verify your identity immediately click link below",
]

vec = tfidf.transform([bersihkan_teks(t) for t in contoh_teks])
preds = svm_model.predict(vec)

for teks, pred in zip(contoh_teks, preds):
    print(f"[{pred:5s}] {teks[:60]}...")

if len(set(preds)) == 1:
    print("\nPERINGATAN: model menjawab satu kelas yang sama untuk semua contoh - cek ulang data/training!")
else:
    print("\nModel berhasil membedakan kelas pada contoh di luar dataset.")

### 11. Export Model & Vectorizer

Simpan HANYA jika hasil evaluasi (langkah 9) dan sanity check (langkah 10) menunjukkan performa yang wajar.

In [ ]:
import joblib

model_path = '/content/drive/MyDrive/Semester 4/Machine Learning/Final Project/models/spam_model_svm.pkl'
tfidf_path = '/content/drive/MyDrive/Semester 4/Machine Learning/Final Project/models/tfidf_vectorizer.pkl'

joblib.dump(svm_model, model_path)
joblib.dump(tfidf, tfidf_path)

print("Model dan TF-IDF Vectorizer berhasil disimpan!")
print(f"Lokasi Model : {model_path}")
print(f"Lokasi TF-IDF: {tfidf_path}")